# TP 2 : FILTRAGE DE NYQUIST ET TRANSMISSION EN BANDE LIMITÉE

**Établissement** : USTHB - Faculté de Genie Electrique  
**Département** : 4ème année GEI, ELN  
**Année universitaire** : 2025-2026  
**Chargé de TP** : Mr. MEFTAH

---
### ⚠️ INSTRUCTIONS POUR LES ÉTUDIANTS :
1. **Complétez** toutes les zones de code marquées `### A COMPLETER ###`.
2. **Déduisez** les valeurs masquées notées `α = ?` ou `Instant = ?` et remplacez-les dans le code.
3. **Répondez** aux questions textuelles posées directement dans les cellules Markdown prévues à cet effet.
4. N'oubliez pas de **remplir le Tableau 2.1** dans votre compte-rendu.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Définition de la classe et des fonctions de base

In [9]:
class FiltrageNyquist:
    def __init__(self, Rb=1000, fs=8000, Nbits=200):
        self.Rb = Rb
        self.fs = fs
        self.Nbits = Nbits
        self.Tb = 1/Rb
        
        # QUESTION : Quelle est la relation entre Ns, fs et Tb ?
        self.Ns = int(fs * self.Tb)  
        if self.Ns == 0:
            self.Ns = 1
            
        self.t = np.arange(0, Nbits * self.Tb, 1/fs)
        
    def sinc(self, x):
        x = np.asarray(x, dtype=float)
        result = np.ones_like(x)
        mask = np.abs(x) > 1e-10
        result[mask] = np.sin(np.pi * x[mask]) / (np.pi * x[mask])
        return result
    
    def filtre_cosinus_sureleve(self, alpha, L=10):
        N_pts = 2 * L * self.Ns + 1
        t_imp = np.linspace(-L*self.Tb, L*self.Tb, N_pts)
        h = np.zeros(N_pts, dtype=float)
        
        for i, t_val in enumerate(t_imp):
            t_norm = t_val / self.Tb
            
            if abs(t_val) < 1e-12:
                h[i] = 1.0 / self.Tb
            elif alpha > 1e-10 and abs(abs(2*alpha*t_norm) - 1.0) < 1e-10:
                h[i] = (np.pi / (4 * self.Tb)) * self.sinc(1/(2*alpha))
            else:
                sinc_val = self.sinc(t_norm)
                cos_val = np.cos(np.pi * alpha * t_norm)
                den = 1 - (2*alpha*t_norm)**2
                
                if abs(den) < 1e-10:
                    h[i] = (np.pi / (4 * self.Tb)) * self.sinc(1/(2*alpha))
                else:
                    h[i] = (sinc_val * cos_val / den) / self.Tb
                    
        h = h / np.sqrt(np.sum(h**2))
        return t_imp, h
    
    def reponse_frequentielle(self, h, Nfft=2048):
        H = np.fft.fft(h, Nfft)
        H = np.fft.fftshift(H)
        f = np.fft.fftshift(np.fft.fftfreq(Nfft, 1/self.fs))
        return f, 20*np.log10(np.abs(H) + 1e-12)
    
    def generer_sequence(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        bits = np.random.randint(0, 2, self.Nbits)
        symbols = 2*bits - 1
        return bits, symbols.astype(float)
    
    def sur_echantillonnage(self, symbols):
        signal_sur = np.zeros(len(symbols) * self.Ns)
        signal_sur[::self.Ns] = symbols
        return signal_sur
    
    def filtrer_signal(self, signal_in, h):
        return np.convolve(signal_in, h, mode='same')

> **Réponse aux questions :**
> - À quoi correspond physiquement la fonction mathématique `sinc` ? 
> *[Votre réponse ici]*
> 
> - Quelle est la formule théorique de $h(t)$ pour le filtre en cosinus surélevé ?
> *[Votre réponse ici]*
> 
> - Pourquoi obtient-on un `sinc` pur quand $\alpha = 0$ ?
> *[Votre réponse ici]*
> 
> - Pourquoi ajoute-t-on `1e-12` dans le calcul de la réponse fréquentielle en dB ?
> *[Votre réponse ici]*
> 
> - Quel type de mapping est utilisé dans `generer_sequence` ?
> *[Votre réponse ici]*

## 2. Fonctions d'analyse (Diagramme de l'œil et IES)

In [10]:
    def diagramme_oeil(self, signal_rx, alpha):
        N_oeil = 2 * self.Ns
        if len(signal_rx) < N_oeil + 20*self.Ns:
            print("Signal trop court pour le diagramme de l'œil")
            return
            
        plt.figure(figsize=(12, 6))
        start_idx = 15 * self.Ns
        count = 0
        
        for i in range(start_idx, len(signal_rx) - N_oeil, self.Ns):
            segment = signal_rx[i:i + N_oeil]
            temps_oeil = np.linspace(-1, 1, len(segment))
            plt.plot(temps_oeil, segment, 'b-', alpha=0.15, linewidth=0.5)
            count += 1
            if count > 100:
                break
        
        plt.axvline(x=0, color='r', linestyle='--', linewidth=1.5, alpha=0.7, label='### A COMPLETER ###')
        plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
        plt.title('### A COMPLETER ### - α = ?')
        plt.xlabel('### A COMPLETER ###')
        plt.ylabel('### A COMPLETER ###')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xlim(-1, 1)
        plt.tight_layout()
        plt.show()
    
    def calcul_ouverture_ies(self, signal_rx, decalage_echantillons, symbols):
        start_sym = 20
        end_sym = min(self.Nbits - 20, len(signal_rx) // self.Ns - 5)
        
        if end_sym <= start_sym + 10:
            return 0, 1.0, np.array([])
        
        indices = []
        for k in range(start_sym, end_sym):
            idx = k * self.Ns + decalage_echantillons
            if 0 <= idx < len(signal_rx):
                indices.append(idx)
        
        if len(indices) < 10:
            return 0, 1.0, np.array([])
        
        indices = np.array(indices)
        echantillons = signal_rx[indices]
        symboles_ref = symbols[start_sym:start_sym + len(echantillons)]
        
        mask_pos = symboles_ref > 0
        mask_neg = symboles_ref < 0
        
        ech_pos = echantillons[mask_pos]
        ech_neg = echantillons[mask_neg]
        
        if len(ech_pos) < 5 or len(ech_neg) < 5:
            return 0, 1.0, echantillons
        
        min_pos = np.min(ech_pos)
        max_neg = np.max(ech_neg)
        ouverture = max(0, min_pos - max_neg)
        
        max_pos = np.max(ech_pos)
        min_neg = np.min(ech_neg)
        plage_totale = max_pos - min_neg
        
        if plage_totale > 1e-10:
            ies = 1.0 - ouverture / plage_totale
        else:
            ies = 1.0
        
        return ouverture, ies, echantillons
    
    def calcul_marge_bruit(self, ouverture):
        if ouverture <= 1e-10:
            return -40.0
        return 20 * np.log10(ouverture / 2)
    
    def calcul_penalite_snr(self, ouv_actuelle, ouv_reference):
        if ouv_reference <= 1e-10 or ouv_actuelle <= 1e-10:
            return float('inf')
        return 20 * np.log10(ouv_reference / ouv_actuelle)

> **Réponse aux questions :**
> - Que représente le diagramme de l'œil et comment l'interpréter ?
> *[Votre réponse ici]*
> 
> - Qu'est-ce que l'IES et quel est son lien avec le critère de Nyquist ?
> *[Votre réponse ici]*
> 
> - Quel est le lien mathématique entre l'ouverture de l'œil et la marge au bruit ?
> *[Votre réponse ici]*

## 3. Définition des Manipulations

In [11]:
    # --- INJECTION DES METHODES DANS LA CLASSE ---
    # (Dans un notebook, on attache les methodes dynamiquement)
FiltrageNyquist.diagramme_oeil = diagramme_oeil
FiltrageNyquist.calcul_ouverture_ies = calcul_ouverture_ies
FiltrageNyquist.calcul_marge_bruit = calcul_marge_bruit
FiltrageNyquist.calcul_penalite_snr = calcul_penalite_snr

def analyser_alpha(self, alphas=[0.25, 0.5, 0.75]):
    results = {}
    bits, symbols = self.generer_sequence(seed=42)
    
    for idx, alpha in enumerate(alphas):
        print(f"\n{'='*50}")
        print(f"Analyse pour Filtre {idx+1} (α = ?)")
        print('='*50)
        
        try:
            t_imp, h = self.filtre_cosinus_sureleve(alpha)
            
            signal_sur = self.sur_echantillonnage(symbols)
            signal_tx = self.filtrer_signal(signal_sur, h)
            signal_canal = signal_tx.copy()
            
            signal_rx = self.filtrer_signal(signal_canal, h)
            
            self.diagramme_oeil(signal_rx, alpha)
            
            instant_optimal = 0 
            ouverture, ies, echantillons = self.calcul_ouverture_ies(
                signal_rx, instant_optimal, symbols
            )
            
            marge_db = self.calcul_marge_bruit(ouverture)
            
            f, H_dB = self.reponse_frequentielle(h)
            H_linear = 10**(H_dB/20)
            H_max = np.max(H_linear)
            
            idx_3db = np.where(H_linear >= 0.707 * H_max)[0]
            BW = f[idx_3db[-1]] - f[idx_3db[0]] if len(idx_3db) > 0 else 0
            
            print(f"Ouverture de l'œil: {ouverture:.4f} V")
            print(f"IES mesuré: {ies:.4f}")
            print(f"Marge au bruit: {marge_db:.2f} dB")
            print(f"Bande passante à -3dB: {BW:.1f} Hz")
            
            results[alpha] = {
                'h': h,
                't_imp': t_imp,
                'signal_rx': signal_rx,
                'symbols': symbols,
                'ies': ies,
                'ouverture': ouverture,
                'marge_db': marge_db,
                'bande_passante': BW,
                'echantillons': echantillons
            }
            
        except Exception as e:
            print(f"Erreur pour le filtre {idx+1}: {e}")
            continue
            
    return results

FiltrageNyquist.analyser_alpha = analyser_alpha

def analyser_synchronisation(self, alpha=0.5):
    print(f"\n{'='*60}")
    print(f"ANALYSE DE LA SYNCHRONISATION (α = ?)")
    print('='*60)
    
    try:
        t_imp, h = self.filtre_cosinus_sureleve(alpha)
        bits, symbols = self.generer_sequence(seed=123)
        signal_sur = self.sur_echantillonnage(symbols)
        signal_tx = self.filtrer_signal(signal_sur, h)
        signal_rx = self.filtrer_signal(signal_tx, h)
        
        # INSTANTS MASQUÉS POUR L'ÉTUDIANT
        decalages = [0, self.Ns//4, self.Ns//2, 3*self.Ns//4]
        labels = ['Instant = ? (1)', 'Instant = ? (2)', 'Instant = ? (3)', 'Instant = ? (4)']
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        
        ouv_ref, _, _ = self.calcul_ouverture_ies(signal_rx, 0, symbols)
        
        print("\n--- Résultats par instant d'échantillonnage ---")
        print(f"{'Instant':<20} {'Ouverture (V)':<15} {'IES':<12} {'Pénalité (dB)':<15}")
        print("-" * 62)
        
        for i, (decalage, label) in enumerate(zip(decalages, labels)):
            ouverture, ies, echantillons = self.calcul_ouverture_ies(signal_rx, decalage, symbols)
            
            if len(echantillons) == 0:
                axes[i].set_title(f'{label} - Pas de données')
                continue
            
            penalite = self.calcul_penalite_snr(ouverture, ouv_ref)
            
            pen_str = f"{penalite:.2f}" if penalite != float('inf') else "∞"
            print(f"{label:<20} {ouverture:<15.4f} {ies:<12.4f} {pen_str:<15}")
            
            n_plot = min(50, len(echantillons))
            axes[i].plot(echantillons[:n_plot], 'bo-', alpha=0.7, markersize=3)
            axes[i].axhline(y=1, color='r', linestyle='--', alpha=0.5, label='### A COMPLETER ###')
            axes[i].axhline(y=-1, color='r', linestyle='--', alpha=0.5, label='### A COMPLETER ###')
            axes[i].axhline(y=0, color='k', linestyle='-', alpha=0.3)
            
            axes[i].set_title(f'### A COMPLETER ### - {label}\nOuverture: {ouverture:.3f} V, IES: {ies:.3f}')
            axes[i].set_xlabel('### A COMPLETER ###')
            axes[i].set_ylabel('### A COMPLETER ###')
            axes[i].set_ylim(-2, 2)
            axes[i].grid(True, alpha=0.3)
            axes[i].legend(loc='upper right', fontsize=7)
        
        plt.suptitle('### A COMPLETER ### - α = ?', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Erreur: {e}")

FiltrageNyquist.analyser_synchronisation = analyser_synchronisation

def analyser_robustesse_alpha(self, alphas=[0.25, 0.5, 0.75]):
    print(f"\n{'='*60}")
    print("ANALYSE COMPARATIVE DE LA ROBUSTESSE")
    print('='*60)
    
    print(f"\n{'Filtre':<10} {'Ouv. opt. (V)':<15} {'Ouv. décalé (V)':<15} {'Pénalité (dB)':<15}")
    print("-" * 55)
    
    for idx, alpha in enumerate(alphas):
        try:
            t_imp, h = self.filtre_cosinus_sureleve(alpha)
            bits, symbols = self.generer_sequence(seed=456)
            signal_sur = self.sur_echantillonnage(symbols)
            signal_tx = self.filtrer_signal(signal_sur, h)
            signal_rx = self.filtrer_signal(signal_tx, h)
            
            ouv_opt, ies_opt, _ = self.calcul_ouverture_ies(signal_rx, 0, symbols)
            ouv_tb4, ies_tb4, _ = self.calcul_ouverture_ies(signal_rx, self.Ns//4, symbols)
            penalite = self.calcul_penalite_snr(ouv_tb4, ouv_opt)
            
            pen_str = f"{penalite:.2f}" if penalite != float('inf') else "∞"
            print(f"Filtre {idx+1}   {ouv_opt:<15.4f} {ouv_tb4:<15.4f} {pen_str:<15}")
            
        except Exception as e:
            print(f"Filtre {idx+1}   Erreur: {e}")
            continue

FiltrageNyquist.analyser_robustesse_alpha = analyser_robustesse_alpha

IndentationError: expected an indented block after function definition on line 8 (319400791.py, line 9)

> **Réponse aux questions :**
> - Pourquoi le signal est-il filtré deux fois (émission et réception) avec le même filtre ?
> *[Votre réponse ici]*
> 
> - Pourquoi a-t-on choisi le seuil `0.707` pour calculer la bande passante ?
> *[Votre réponse ici]*
> 
> - Comment l'IES varie-t-elle avec l'instant d'échantillonnage ? Pourquoi ?
> *[Votre réponse ici]*

## 4. Exécution du TP (Programme Principal)

In [ ]:
print("="*70)
print("TP 2 : FILTRAGE DE NYQUIST")
print("USTHB - 4ème GEI, ELN - Communications Numériques")
print("="*70)

sim = FiltrageNyquist(Rb=1000, fs=8000, Nbits=200)
alphas = [0.25, 0.5, 0.75]

print("\n" + "="*70)
print("MANIPULATION 1 : CARACTÉRISATION DES FILTRES")
print("="*70)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
colors = ['blue', 'red', 'green']

print(f"\n{'Filtre':<10} {'Bande -3dB (Hz)':<18}")
print("-" * 30)

for i, alpha in enumerate(alphas):
    t_imp, h = sim.filtre_cosinus_sureleve(alpha)
    f, H_dB = sim.reponse_frequentielle(h)
    
    ax1.plot(t_imp*1000, h, linewidth=2, color=colors[i], label='α = ?')
    ax2.plot(f, H_dB, linewidth=2, color=colors[i], label='α = ?')
    
    H_linear = 10**(H_dB/20)
    H_max = np.max(H_linear)
    idx_3db = np.where(H_linear >= 0.707 * H_max)[0]
    BW = f[idx_3db[-1]] - f[idx_3db[0]] if len(idx_3db) > 0 else 0
    
    print(f"Filtre {i+1}   {BW:<18.1f}")

ax1.set_title('### A COMPLETER ###')
ax1.set_xlabel('### A COMPLETER ###')
ax1.set_ylabel('### A COMPLETER ###')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.axhline(y=0, color='k', linestyle='-', alpha=0.3)
for k in range(-5, 6):
    ax1.axvline(x=k*sim.Tb*1000, color='gray', linestyle=':', alpha=0.4)

ax2.set_title('### A COMPLETER ###')
ax2.set_xlabel('### A COMPLETER ###')
ax2.set_ylabel('### A COMPLETER ###')
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.set_xlim(-2000, 2000)
ax2.set_ylim(-60, 5)
ax2.axhline(y=-3, color='k', linestyle='--', alpha=0.5, label='### A COMPLETER ###')

plt.tight_layout()
plt.show()

> **Réponse aux questions :**
> - Que se passe-t-il aux instants $t = k \cdot T_b$ dans la réponse impulsionnelle ?
> *[Votre réponse ici]*
> 
> - Comment évolue la bande passante avec $\alpha$ ?
> *[Votre réponse ici]*

In [ ]:
print("\n" + "="*70)
print("MANIPULATION 2 : ANALYSE DE LA CHAÎNE DE TRANSMISSION")
print("="*70)
results = sim.analyser_alpha(alphas=alphas)

print("\n" + "="*70)
print("MANIPULATION 3 : OPTIMISATION ET ROBUSTESSE")
print("="*70)
sim.analyser_synchronisation(alpha=0.5)
sim.analyser_robustesse_alpha(alphas=alphas)

## 5. Synthèse et Recommandations

In [ ]:
print("\n" + "="*70)
print("TABLEAU RÉCAPITULATIF")
print("="*70)

print(f"\n{'Métrique':<25} ", end="")
for i in range(len(alphas)):
    print(f"{f'Filtre {i+1} (α = ?)':<18}", end="")
print("\n" + "-" * 79)

for key, name in [('ouverture', 'Ouverture œil (V)'), 
                  ('ies', 'IES mesuré'), 
                  ('marge_db', 'Marge au bruit (dB)'), 
                  ('bande_passante', 'Bande passante (Hz)')]:
    print(f"{name:<25} ", end="")
    for alpha in alphas:
        if alpha in results:
            val = results[alpha][key]
            fmt = ".2f" if key == 'marge_db' or key == 'bande_passante' else ".4f"
            print(f"{val:<18{fmt}}", end="")
        else:
            print(f"{'N/A':<18}", end="")
    print()

print("-" * 79)

> **Conclusion Finale :**
> Quel facteur $\alpha$ recommanderiez-vous pour :
> - Une liaison satellite (bande très limitée, synchronisation parfaite) ?
> *[Votre réponse ici]*
> 
> - Une liaison mobile (canal bruité, synchronisation imparfaite) ?
> *[Votre réponse ici]*